In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore")
%matplotlib inline

sns.set(style="darkgrid")

# Выбор задачи
Был выбран случайный лес из 2 лабы.

In [4]:
# Загрузка

DATA_PATH= "train.csv"
data = pd.read_csv(DATA_PATH, sep=",")
print(data.shape)
print("Типы признаков:")
print(data.dtypes.to_string())

(891, 9)
Типы признаков:
Id            int64
Pclass        int64
Sex             str
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked        str
Survived      int64


In [5]:
# Подготовка данных
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder 
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE=42
X = data.drop(columns=['Survived'])
y = data['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
skewed = ['Fare']
num_cols = ['Age', 'SibSp', 'Parch']
cat_cols = ['Sex', 'Pclass', 'Embarked']
preprocess = ColumnTransformer([
    ('num', Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median"))
        ]
            ), num_cols),
    ('cat', Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown='ignore'))
            ]), cat_cols),
    
])
random_forest = Pipeline(
    [("preprocess", preprocess),
      ("tree",
       RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    )]
)

In [6]:
#baseline
from sklearn.model_selection import StratifiedKFold, cross_validate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
baseline = Pipeline(
    [("preprocess", preprocess),
      ("tree",
       RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, max_depth=10, min_samples_leaf=4, n_estimators=300)
    )]
)
scores = cross_validate(
    baseline, X_train, y_train, cv=cv,
    scoring="f1",
    return_train_score=True, n_jobs=-1
)
print(f"Среднее: {scores["test_score"].mean()}\nСтандартное отклонение {scores["test_score"].std()}")
print("F1 метрики")
print(scores["test_score"])

Среднее: 0.6966641841778621
Стандартное отклонение 0.036929903372152406
F1 метрики
[0.68085106 0.69387755 0.70212766 0.64646465 0.76      ]


# Гиперпараметры
n_estimators, max_depth, min_samples_leaf.
## n_estimators
Количество деревьев в случайном лесе.
Диапазон значений: 100 - 500.
При повышении параметра стабилизируется impurity, уменьшается переобучение, но увеличивается время обучения модели.
## max_depth
Максимальная глубина каждого дерева.
Диапазон значений: 2 - 16.
Слишком маленькая максимальная глубина приводит к underfitting, а слишком большая к overfitting.
## min_samples_leaf
Минимальное количество объектов в листе.
Диапазон значений - 2 - 32.
При росте параметра модель меньше переобучается. При слишком большом параметре происходит underfitting.

# Кросс-валидация
Я возьму 5 folds, это стандартное значение. 142 объекта уйдет на тест, остальное на обучение.
1. Зачем нужна кросс-валидация
Чтобы получить устойчивую оценку обобщения: каждое наблюдение побывает в тесте ровно один раз, а оценка не зависит от удачности одного разбиения.
2. Почему одного train/test недостаточно для подбора
Хуже обобщается. Разница в нескольких объектах хуже влияет на метрику.
3. Что будет при многократном подборе по test
Тест перестает быть независимым. Мы подстраиваемся под тест и получаем завышенную оценку.

In [22]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
import time
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
parameter_grid = {
    "tree__n_estimators": [100, 300, 500],
    "tree__max_depth": [2, 4, 8, 16],
    "tree__min_samples_leaf": [2, 4, 8, 16, 32],
}
prev_time = time.perf_counter()
grid_search = GridSearchCV(
    random_forest, parameter_grid, cv=cv,
    scoring="f1",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
elapsed = time.perf_counter() - prev_time
r = grid_search.cv_results_
print("Лучшие гиперпараметры")
print(grid_search.best_params_)
print("Значение метрики")
print(grid_search.best_score_)
best_model = grid_search.best_estimator_
results = pd.DataFrame(grid_search.cv_results_)
results[[
    "params",
    "mean_test_score",
    "std_test_score",
    "mean_fit_time"
]].sort_values("mean_test_score", ascending=False).head()
print(f"Количество проверенных комбинаций {len(r["params"])}")
print(f"Затрачено времени: {elapsed}")

Лучшие гиперпараметры
{'tree__max_depth': 4, 'tree__min_samples_leaf': 2, 'tree__n_estimators': 100}
Значение метрики
0.7191554578924368
Количество проверенных комбинаций 60
Затрачено времени: 29.25765095799943


In [ ]:
#random search
import time
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint
parameter_dist = {
    "tree__n_estimators": randint(100, 800),
    "tree__max_depth": randint(2, 16),
    "tree__min_samples_leaf": randint(2, 32),
}
random_search= RandomizedSearchCV(
    random_forest, parameter_dist,
    n_iter=60,                 # сколько конфигураций попробовать
    cv=cv,
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=-1, 
)
prev_time = time.perf_counter()
random_search.fit(X_train, y_train)
elapsed = time.perf_counter() - prev_time
r = random_search.cv_results_
print("Лучшие гиперпараметры")
print(random_search.best_params_)
print("Значение метрики")
print(random_search.best_score_)
best_model = random_search.best_estimator_
results = pd.DataFrame(random_search.cv_results_)
results[[
    "params",
    "mean_test_score",
    "std_test_score",
    "mean_fit_time"
]].sort_values("mean_test_score", ascending=False).head()
print(f"Количество проверенных комбинаций {len(r["params"])}")
print(f"Затрачено времени: {elapsed}")

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'rv_continuous_frozen'